<a href="https://colab.research.google.com/github/12370043/keiichiro/blob/main/%E5%88%86%E5%AD%90%E3%81%AE%E6%B4%BB%E6%80%A7%E4%BA%88%E6%B8%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## プロジェクト概要
このリポジトリでは、RDKitとPyTorch Geometricを用いて、SMILESから分子グラフを構築し、Graph Convolutional Network（GCN）により分子の活性を予測しています。

## 出力
Epoch 1, Loss: 2.1825
Epoch 2, Loss: 1.9691
Epoch 3, Loss: 1.9312
Epoch 4, Loss: 1.9105
Epoch 5, Loss: 1.8993
Epoch 6, Loss: 1.8951
Epoch 7, Loss: 1.8954
Epoch 8, Loss: 1.8982
Epoch 9, Loss: 1.9018
Epoch 10, Loss: 1.9054
Epoch 11, Loss: 1.9083
Epoch 12, Loss: 1.9105
Epoch 13, Loss: 1.9118
Epoch 14, Loss: 1.9124
Epoch 15, Loss: 1.9126
Epoch 16, Loss: 1.9124
Epoch 17, Loss: 1.9120
Epoch 18, Loss: 1.9114
Epoch 19, Loss: 1.9108
Epoch 20, Loss: 1.9102

## 今後の展望
・電荷・結合の種類などの特徴量を追加
・GATやGINへの拡張
・ROC曲線による性能可視化

In [ ]:
# PyTorch + PyTorch Geometric + RDKit
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install rdkit-pypi -q
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.0.1+cu118.html -q
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.0.1+cu118.html -q
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 45.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

# ファイル名を取得し読み込み
import pandas as pd
filename = next(iter(uploaded))
df = pd.read_csv(f"/content/{filename}")
df.head()

Saving gcn_starter_dataset.csv to gcn_starter_dataset (1).csv


,SMILES,Activity,MolWt,LogP,NumHDonors,NumHAcceptors
0,CCO,1,46.069,-0.0014,1,1
1,CC(=O)OC1=CC=CC=C1C(=O)O,0,180.159,1.3101,1,3
2,CCN(CC)CCCC(C)NC1=NC2=CC=CC=C2C(=N1)C3=CC=CC=C3,1,362.521,5.2192,1,4
3,C1=CC=C(C=C1)C=O,0,106.124,1.4991,0,1
4,CCOC(=O)C1=CC=CC=C1Cl,0,184.622,2.5167,0,2


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from rdkit import Chem
from sklearn.model_selection import train_test_split

# SMILES → 分子グラフ変換（今回は原子番号のみ）
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    atoms = mol.GetAtoms()
    edges = []
    for bond in mol.GetBonds():
        edges.append([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
        edges.append([bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()])
    x = torch.tensor([[atom.GetAtomicNum()] for atom in atoms], dtype=torch.float)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

# 分子グラフリストとラベル作成
graphs = []
for _, row in df.iterrows():
    graph = smiles_to_graph(row["SMILES"])
    if graph:
        graph.y = torch.tensor([row["Activity"]], dtype=torch.float)
        graphs.append(graph)

# データ分割
train_graphs, test_graphs = train_test_split(graphs, test_size=0.4, random_state=42)

# GCNモデル構築
class GCN(torch.nn.Module):
    def __init__(self):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(1, 16)
        self.conv2 = GCNConv(16, 8)
        self.fc = torch.nn.Linear(8, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = torch.mean(x, dim=0)  # グラフ全体の特徴に集約
        return self.fc(x)


In [ ]:
# モデル初期化
model = GCN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

# ロス記録用リスト（グラフ可視化にも使える）
losses = []

# 学習ループ（20エポック）
for epoch in range(20):
    model.train()
    total_loss = 0
    for graph in train_graphs:
        optimizer.zero_grad()
        out = model(graph)  # GCNによる活性予測
        loss = criterion(out.view(-1), graph.y)  # 活性ラベルとの誤差計算
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    losses.append(total_loss)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 2.1825
Epoch 2, Loss: 1.9691
Epoch 3, Loss: 1.9312
Epoch 4, Loss: 1.9105
Epoch 5, Loss: 1.8993
Epoch 6, Loss: 1.8951
Epoch 7, Loss: 1.8954
Epoch 8, Loss: 1.8982
Epoch 9, Loss: 1.9018
Epoch 10, Loss: 1.9054
Epoch 11, Loss: 1.9083
Epoch 12, Loss: 1.9105
Epoch 13, Loss: 1.9118
Epoch 14, Loss: 1.9124
Epoch 15, Loss: 1.9126
Epoch 16, Loss: 1.9124
Epoch 17, Loss: 1.9120
Epoch 18, Loss: 1.9114
Epoch 19, Loss: 1.9108
Epoch 20, Loss: 1.9102


Epoch 1 → 3 ------- 2.18 → 1.93 → 急激に減少（急速な初期学習）

Epoch 4 → 9 ------- 約1.89台で安定（収束傾向）

Epoch 10以降 ------- 1.90台前半でほぼ横ばい（改善が止まる）

--> ・構造情報を使って活性をある程度予測できている
    ・Lossが下がっている.モデルが「ある程度」分子構造と活性の関係を学習している


1.91前後で停滞 ------特徴量（原子番号）だけでは情報不足


改善点

・特徴量追加（電荷・結合種類・芳香族性など）

・分子表現を強化し精度向上を狙う

・GATやGINなどのモデルに変更

・より複雑な構造の関係性を学習させる

・評価指標の導入（AUC, Accuracy）
